# LSSTCam FWHM vs AuxTel FWHM vs Tower DIMM vs Portable DIMM

This notebook is part of a large study to understand the baseline expected seeing in the images obtained with the LSSTCam. We know that the image quality obtained with Simonyi and its large camera is affected by several aspects like the optical state, the thermal control system, the environment conditions, dome seeing, atmosphere seeing, etc. It is quite a challenge to separate all these effects. So, part of this analysis require comparison between different instruments.

## Setup Notebook

In [ ]:
day_obs_start = 20251001
day_obs_end = 20260131
wind_telemetry_sampling = "30s"
wind_threshold = 4  # m/s

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import astropy.units as u

from astroplan import Observer
from astropy.coordinates import EarthLocation
from astropy.time import Time, TimeDelta
from datetime import datetime, timedelta, date

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.layouts import column, row
from bokeh.models import (
    Arrow,
    BoxZoomTool,
    ColumnDataSource,
    DataRange1d,
    HoverTool,
    Legend,
    LegendItem,
    Label,
    OpenHead,
    PanTool,
    Range1d,
    ResetTool,
    SaveTool,
    Span,
    Title,
    WheelZoomTool,
)

from lsst_efd_client import EfdClient
from lsst.summit.utils import (
    ConsDbClient,
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)
from lsst.summit.utils.dateTime import getDayObsStartTime, getDayObsEndTime, calcNextDay
from lsst.summit.utils.efdUtils import getEfdData

from scipy.stats import gaussian_kde

from tqdm.notebook import tqdm


# Update default colors to increase contrast
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=["#0072B2", "#D55E00", "#009E73"])

# Make sure Bokeh initializes properly
output_notebook()

# Some constants used in the notebook
SIGMA_TO_FWHM = 2 * np.sqrt(2 * np.log(2))
PIXEL_SCALE_ARCSEC = 0.2  # arcsec / pixel (LSSTCam)
SAL_INDEX_TOWER_DIMM = 1
SAL_INDEX_PORTABLE_DIMM = 2
SAL_INDEX_WEATHER_TOWER = 301

# Create an EFD Client
efd_client = EfdClient("usdf_efd")

# Required to use ConsDb, following documentation above
os.environ["no_proxy"] += ",.consdb"

# Initialize ConsDb
cdb_client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")

# Rubin Observatory location
rubin_observatory = Observer(
    location=EarthLocation.from_geodetic(
        lon=-70.7494 * u.deg,
        lat=-30.2444 * u.deg,
        height=2663 * u.m,
    ),
    name="Rubin",
    timezone="Chile/Continental",
)

# Define the time stamps
t_start = getDayObsStartTime(day_obs_start)
t_end = getDayObsEndTime(day_obs_end)
print(f"Running analysis from {t_start} to {t_end}")

## Query the data

### Tower DIMM and Portable DIMM

Here we perform a single query to download data from both DIMMS.

In [ ]:
# Query all the DIMM data - It will contain data from both
#  Tower DIMM (sal index 1) and Portable DIMM (sal index 2)
df_dimm_measurement = getEfdData(
    efd_client,
    "lsst.sal.DIMM.logevent_dimmMeasurement",
    columns=["salIndex", "fwhm", "secz"],
    begin=t_start,
    end=t_end,
)

# Drop invalid rows
df_dimm_measurement.dropna(inplace=True)

# Filter out seeing (FWHM) measurements that are too high
df_dimm_measurement = df_dimm_measurement[df_dimm_measurement["fwhm"] < 5]

# Correct seeing measurements due to airmass (secz)
df_dimm_measurement["fwhm_z"] = df_dimm_measurement["fwhm"] * df_dimm_measurement[
    "secz"
] ** (-3 / 5)

# Split data per DIMM
sal_index = df_dimm_measurement["salIndex"]
df_tower_dimm = df_dimm_measurement[sal_index == SAL_INDEX_TOWER_DIMM]
df_portable_dimm = df_dimm_measurement[sal_index == SAL_INDEX_PORTABLE_DIMM]

print(
    f"Queried data from both DIMMs between {day_obs_start} and {day_obs_end}.\n"
    f"  Got {df_tower_dimm.index.size} data points for the Tower DIMM.\n"
    f"  Got {df_portable_dimm.index.size} data points for the Portable DIMM.\n"
)

### LSSTCam Median FWHM at Zenith and for 500 nm

In [ ]:
def convert_psf_sigma_to_fwhm(
    psf_sigma: pd.Series, airmass: pd.Series, band_p: pd.Series
) -> pd.Series:
    """
    Convert PSF sigma to FWHM.

    Parameters
    ----------
    psf_sigma : pd.Series
        The PSF sigma values in pixels.
    airmass : pd.Series
        The airmass values.
    band_p : pd.Series
        The physical filter names.

    Returns
    -------
    pd.Series
        The corresponding FWHM values at zenith for 500 nm.
    """
    # Convert PSF sigma (pixels) -> FWHM (arcsec)
    # NOTE: psf_sigma is the median sigma from visit1_quicklook.
    psf_fwhm = psf_sigma * SIGMA_TO_FWHM * PIXEL_SCALE_ARCSEC

    # Apply bandpass and airmass corrections
    airmass_correction = airmass.apply(getAirmassSeeingCorrection)
    bandpass_correction = band_p.apply(getBandpassSeeingCorrection)

    fwhm_zenith_500nm = psf_fwhm * airmass_correction * bandpass_correction
    return fwhm_zenith_500nm


# Let's query data from ConsDB
cdb_lsstcam_query = f"""
    SELECT
        e.seq_num AS seq,
        e.day_obs,
        q.physical_rotator_angle,
        e.altitude,
        e.airmass,
        e.obs_start,
        e.obs_end,
        e.focus_z,
        e.observation_reason,
        e.physical_filter as band_p,
        e.band,
        e.wind_speed,
        e.wind_dir,
        q.psf_sigma_median,
        q.aos_fwhm
    FROM
        cdb_lsstcam.visit1_quicklook AS q,
        cdb_lsstcam.exposure AS e
    WHERE
        q.visit_id = e.exposure_id
        AND (e.img_type = 'science')
        AND e.day_obs >= {int(day_obs_start)}
        AND e.day_obs <= {int(day_obs_end)}
"""

df_lsstcam = cdb_client.query(cdb_lsstcam_query).to_pandas()

# Drop any data with the following criteria
df_lsstcam = df_lsstcam[df_lsstcam["airmass"] != 0]
df_lsstcam = df_lsstcam[df_lsstcam["band_p"] != "none"]
df_lsstcam = df_lsstcam.reset_index(drop=True)  # Reset index after filtering

# Robust datetime parsing for mixed ISO8601 strings
df_lsstcam["obs_start"] = pd.to_datetime(
    df_lsstcam["obs_start"], format="ISO8601", errors="coerce"
)
df_lsstcam["obs_end"] = pd.to_datetime(
    df_lsstcam["obs_end"], format="ISO8601", errors="coerce"
)

# In theory, this convertion should be already done in ConsDB. However,
#  this column has not bee populated to this date. So we need to update
#  our table manually.
df_lsstcam["fwhm_zenith_500nm_median"] = convert_psf_sigma_to_fwhm(
    psf_sigma=df_lsstcam["psf_sigma_median"],
    airmass=df_lsstcam["airmass"],
    band_p=df_lsstcam["band_p"],
)

# Print some statistics
print(
    f"Got {df_lsstcam.index.size} science exposures from {day_obs_start} to {day_obs_end}"
)

### LATISS Median FWHM at Zenith and for 500 nm

In [ ]:
cdb_latiss_query = f"""
    SELECT
        e.exposure_id,
        e.day_obs,
        e.obs_start,
        e.obs_end,
        e.airmass,
        e.physical_filter,
        e.band,
        e.target_name,
        e.wind_speed,
        e.wind_dir,
        e.dimm_seeing,
        q.psf_sigma,
        q.seeing_zenith_500nm,
        q.psf_area
    FROM
        cdb_latiss.visit1 AS v
    JOIN
        cdb_latiss.exposure AS e ON v.visit_id = e.exposure_id
    JOIN
        cdb_latiss.visit1_quicklook AS q ON v.visit_id = q.visit_id
    WHERE
        e.day_obs >= {int(day_obs_start)}
        AND e.day_obs <= {int(day_obs_end)}
        AND e.img_type = 'science'
"""

df_latiss = cdb_client.query(cdb_latiss_query).to_pandas()
print(
    f"Got {df_latiss.index.size} science exposures from {day_obs_start} to {day_obs_end} for LATISS"
)

# Drop any data with the following criteria
df_latiss = df_latiss[df_latiss["airmass"] != 0]
df_latiss = df_latiss[df_latiss["physical_filter"] != "none"]
df_latiss = df_latiss.reset_index(drop=True)  # Reset index after filtering

# Robust datetime parsing for mixed ISO8601 strings
df_latiss["obs_start"] = pd.to_datetime(
    df_latiss["obs_start"], format="ISO8601", errors="coerce"
)
df_latiss["obs_end"] = pd.to_datetime(
    df_latiss["obs_end"], format="ISO8601", errors="coerce"
)

### Wind Speed and Direction from Weather Tower

The wind speed is a telemetry which has a huge volume of data.
Because of that, we need to downsample the telemetry data using the average per 

In [ ]:
# Calculate all the day obs
list_of_day_obs = []
d = day_obs_start

while d < day_obs_end:
    list_of_day_obs.append(d)
    d = calcNextDay(d)

# Get a placeholder for our tables
frames = []

# Loop through all day obs
with tqdm(list_of_day_obs, desc="Querying nights") as progress_bar:
    for day_obs in progress_bar:

        # Print in the progress bar
        progress_bar.set_postfix(current_value=f"{day_obs:.0f}")

        # Get time stamps
        query_t_start = getDayObsStartTime(day_obs)
        query_t_end = getDayObsEndTime(day_obs)

        # Get the beginning and the end of a night
        t = getDayObsStartTime(day_obs)
        t_night_begin = rubin_observatory.twilight_evening_astronomical(t, "next")
        t_night_end = rubin_observatory.twilight_morning_astronomical(t, "next")

        # We have lots of ESS sensors.
        # Let's use an InfluxDB query to select only data from the weather tower.
        wind_query = f"""
            SELECT "direction", "speed" 
            FROM "efd"."autogen"."lsst.sal.ESS.airFlow" 
            WHERE 
                time > '{t_night_begin.utc.isot}Z' 
                AND time < '{t_night_end.utc.isot}Z' 
                AND salIndex = {SAL_INDEX_WEATHER_TOWER} 
        """

        # And we query the data per chunks
        df_chunk = await efd_client.influx_client.query(wind_query)

        # The query above returns a dict instead of a data frame when there is no data.
        if not isinstance(df_chunk, dict):
            frames.append(df_chunk)

        # Let's get our wind statistics right
        # Ensure time index
        df_chunk = df_chunk.sort_index()

        # Convert direction to radians
        angles = np.deg2rad(df_chunk["direction"])

        # Build temporary dataframe for trig
        df_tmp = df_chunk.copy()
        df_tmp["sin"] = np.sin(angles)
        df_tmp["cos"] = np.cos(angles)

        # Resample
        df_resampled = df_tmp.resample(wind_telemetry_sampling).agg(
            {
                "sin": "mean",
                "cos": "mean",
                "speed": "mean",
            }
        )

        # Reconstruct circular mean
        df_resampled["direction"] = np.rad2deg(
            np.arctan2(df_resampled["sin"], df_resampled["cos"])
        )

        # Normalize to [0, 360)
        df_resampled["direction"] = df_resampled["direction"] % 360

        # Keep only what you need
        df_final = df_resampled[["direction", "speed"]].dropna()
        frames.append(df_final)


# Put everything together
df_ess_airflow = pd.concat(frames)

## Match the data

### Fill DIMM with Wind telemetry

In [ ]:
# Sort both by time index (required for merge_asof)
df_wind = df_ess_airflow[["direction", "speed"]].sort_index()
df_tower = df_tower_dimm.sort_index()
df_portable = df_portable_dimm.sort_index()

tol = pd.Timedelta("30s")

df_tower_wind = pd.merge_asof(
    df_tower,
    df_wind,
    left_index=True,
    right_index=True,
    tolerance=tol,
    direction="nearest",
    suffixes=("", "_wind"),
)

df_portable_wind = pd.merge_asof(
    df_portable,
    df_wind,
    left_index=True,
    right_index=True,
    tolerance=tol,
    direction="nearest",
    suffixes=("", "_wind"),
)

# Clear the data
df_tower_wind.dropna(inplace=True)
df_portable_wind.dropna(inplace=True)

df_tower_wind = df_tower_wind.rename(columns={"speed": "wind_speed"}).sort_index()
df_tower_wind = df_tower_wind.rename(
    columns={"direction": "wind_direction"}
).sort_index()

df_portable_wind = df_portable_wind.rename(columns={"speed": "wind_speed"}).sort_index()
df_portable_wind = df_portable_wind.rename(
    columns={"direction": "wind_direction"}
).sort_index()

### Fill Exposures with DIMM/Wind Data

In [ ]:
def match_dimm_to_visits(df_visits, df_dimm, dimm_col="fwhm_z", prefix="tower_dimm"):
    """Compute mean/std of DIMM seeing within each visit's time window.

    Parameters
    ----------
    df_visits : pd.DataFrame
        Must have 'obs_start' and 'obs_end' columns (datetime).
    df_dimm : pd.DataFrame
        DIMM data with DatetimeIndex and ``dimm_col`` column.
    prefix : str
        Column name prefix for the output.

    Returns
    -------
    df_visits with new columns: {prefix}_mean, {prefix}_std, {prefix}_n
    """
    df = df_visits.copy()

    # Ensure sorted DIMM index
    dimm = df_dimm[[dimm_col]].sort_index().dropna()
    dimm_times = dimm.index.values  # datetime64
    dimm_vals = dimm[dimm_col].values

    obs_start = pd.to_datetime(df["obs_start"]).values
    obs_end = pd.to_datetime(df["obs_end"]).values

    means = np.empty(len(df))
    stds = np.empty(len(df))
    counts = np.empty(len(df), dtype=int)

    for i in range(len(df)):
        i_start = np.searchsorted(dimm_times, obs_start[i], side="left")
        i_end = np.searchsorted(dimm_times, obs_end[i], side="right")
        chunk = dimm_vals[i_start:i_end]

        if len(chunk) > 0:
            means[i] = chunk.mean()
            stds[i] = chunk.std()
            counts[i] = len(chunk)
        else:
            means[i] = np.nan
            stds[i] = np.nan
            counts[i] = 0

    df[f"{prefix}_mean"] = means
    df[f"{prefix}_std"] = stds
    df[f"{prefix}_n"] = counts
    return df

In [ ]:
df_lsstcam = match_dimm_to_visits(
    df_lsstcam, df_tower_dimm, dimm_col="fwhm_z", prefix="tower_dimm"
)
df_lsstcam = match_dimm_to_visits(
    df_lsstcam, df_portable_dimm, dimm_col="fwhm_z", prefix="portable_dimm"
)

## Plot the data

### Wind Speed/Direction Histograms

We want to know what is the predominant wind in our nights. 

In [ ]:
def wind_rose(
    df: pd.DataFrame,
    speed_col: str = "speed",
    direction_col: str = "direction",
    n_dir_bins: int = 16,
    speed_bins: list | None = None,
    figsize: tuple = (9, 9),
    cmap: str = "viridis",
    title: str = "Wind Rose — ESS Weather Tower",
    edgecolor: str = "white",
    linewidth: float = 0.3,
):
    """Stacked‑bar wind rose histogram (matplotlib, polar projection).

    Parameters
    ----------
    df : pd.DataFrame
        Must contain ``speed_col`` (m/s) and ``direction_col`` (degrees, 0–360).
    n_dir_bins : int
        Number of angular slices (16 → 22.5° bins).
    speed_bins : list[float] | None
        Edges for the speed categories.  ``None`` → automatic quintile‑based bins.
    """
    data = df[[speed_col, direction_col]].dropna().copy()

    # ── speed bins ──────────────────────────────────────────────
    if speed_bins is None:
        quantiles = np.quantile(data[speed_col], [0, 0.2, 0.4, 0.6, 0.8, 1.0])
        speed_bins = np.unique(np.round(quantiles, 1))
        # Ensure at least two edges
        if len(speed_bins) < 3:
            speed_bins = np.linspace(data[speed_col].min(), data[speed_col].max(), 6)

    n_speed = len(speed_bins) - 1
    speed_labels = [
        f"{speed_bins[i]:.1f}–{speed_bins[i+1]:.1f} m/s" for i in range(n_speed)
    ]

    # ── direction bins (centered on N = 0°) ─────────────────────
    dir_bin_width = 360.0 / n_dir_bins
    half = dir_bin_width / 2
    dir_edges = np.linspace(-half, 360 - half, n_dir_bins + 1)

    # Shift directions so that the first bin straddles 0°
    shifted = (data[direction_col] + half) % 360 - half
    data["dir_bin"] = pd.cut(shifted, bins=dir_edges, labels=False, include_lowest=True)
    data["spd_bin"] = pd.cut(
        data[speed_col],
        bins=speed_bins,
        labels=False,
        include_lowest=True,
    )

    total = len(data)

    # ── count matrix (direction × speed) ────────────────────────
    counts = np.zeros((n_dir_bins, n_speed))
    for d in range(n_dir_bins):
        for s in range(n_speed):
            counts[d, s] = ((data["dir_bin"] == d) & (data["spd_bin"] == s)).sum()

    freq = 100.0 * counts / total  # percentage

    # ── angular coordinates ─────────────────────────────────────
    bin_centers_deg = np.arange(0, 360, dir_bin_width)
    theta = np.deg2rad(bin_centers_deg)  # meteorological → math handled by axis config
    bar_width = np.deg2rad(dir_bin_width) * 0.95  # tiny gap between bars

    # ── colormap ────────────────────────────────────────────────
    colormap = plt.get_cmap(cmap, n_speed)
    colors = [colormap(i / (n_speed - 1)) for i in range(n_speed)]

    # ── plot ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "polar"}, dpi=120)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)  # clockwise

    bottoms = np.zeros(n_dir_bins)
    bars_per_speed = []
    for s in range(n_speed):
        bars = ax.bar(
            theta,
            freq[:, s],
            width=bar_width,
            bottom=bottoms,
            color=colors[s],
            edgecolor=edgecolor,
            linewidth=linewidth,
            label=speed_labels[s],
        )
        bars_per_speed.append(bars)
        bottoms += freq[:, s]

    # ── radial grid & labels ────────────────────────────────────
    max_pct = bottoms.max()
    tick_step = _nice_tick(max_pct / 4)
    r_ticks = np.arange(tick_step, max_pct + tick_step, tick_step)
    ax.set_yticks(r_ticks)
    ax.set_yticklabels([f"{v:.0f}%" for v in r_ticks], fontsize=8, color="0.4")
    ax.set_rlabel_position(67.5)

    # Cardinal labels
    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(["N", "NE", "E", "SE", "S", "SW", "W", "NW"], fontsize=10)

    # Legend & title
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.05, 1.0),
        title="Wind Speed",
        fontsize=9,
        title_fontsize=10,
        frameon=True,
        fancybox=True,
    )
    ax.set_title(title, pad=24, fontsize=13, weight="bold")

    # ── summary annotation ──────────────────────────────────────
    dominant_dir_idx = bottoms.argmax()
    dominant_dir = bin_centers_deg[dominant_dir_idx]
    mean_speed = data[speed_col].mean()
    median_speed = data[speed_col].median()
    calm_pct = 100.0 * (data[speed_col] < speed_bins[1]).sum() / total

    stats_text = (
        f"N = {total:,}\n"
        f"Dominant dir ≈ {dominant_dir:.0f}°\n"
        f"Mean speed = {mean_speed:.1f} m/s\n"
        f"Median speed = {median_speed:.1f} m/s\n"
        f"Calm (< {speed_bins[1]:.1f} m/s) = {calm_pct:.1f}%"
    )
    fig.text(
        0.96,
        0.2,
        stats_text,
        fontsize=8,
        family="monospace",
        ha="right",
        va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.7", alpha=0.85),
    )

    plt.tight_layout()
    return fig, ax


# ── helper ──────────────────────────────────────────────────────
def _nice_tick(rough):
    """Round *rough* to a 'nice' tick interval (1, 2, 5, 10, …)."""
    if rough <= 0:
        return 1
    mag = 10 ** np.floor(np.log10(rough))
    residual = rough / mag
    if residual <= 1.5:
        return mag
    elif residual <= 3.5:
        return 2 * mag
    elif residual <= 7.5:
        return 5 * mag
    else:
        return 10 * mag

In [ ]:
fig, ax = wind_rose(
    df_ess_airflow, speed_bins=[0, 4, 8, 12, 16, 20], n_dir_bins=32, figsize=(6, 6)
)

### Timeline showing different seeing measurements

In [ ]:
def fwhm_timeline(
    df_tower_dimm: pd.DataFrame,
    df_portable_dimm: pd.DataFrame,
    df_cdb: pd.DataFrame,
    tower_col: str = "fwhm_z",
    portable_col: str = "fwhm_z",
    cam_time_col: str = "obs_end",
    cam_fwhm_col: str = "fwhm_zenith_500nm_median",
    width: int = 1200,
    height: int = 400,
    y_range: tuple = (0, 4),
    title: str = "Seeing Timeline — DIMMs + LSSTCam",
):
    """Interactive timeline of zenith-corrected FWHM from DIMMs and LSSTCam.

    Parameters
    ----------
    df_tower_dimm : pd.DataFrame
        Tower DIMM data with DatetimeIndex and ``tower_col`` column.
    df_portable_dimm : pd.DataFrame
        Portable DIMM data with DatetimeIndex and ``portable_col`` column.
    df_cdb : pd.DataFrame
        LSSTCam ConsDB data with ``cam_time_col`` and ``cam_fwhm_col``.
    """

    # ── prepare sources ─────────────────────────────────────────
    # Strip timezone info to avoid Bokeh datetime64 warnings
    def _strip_tz(idx):
        idx = pd.DatetimeIndex(idx)
        return idx.tz_localize(None) if idx.tz is not None else idx

    src_tower = ColumnDataSource(
        data=dict(
            time=_strip_tz(df_tower_dimm.index),
            fwhm=df_tower_dimm[tower_col].values,
        )
    )

    src_portable = ColumnDataSource(
        data=dict(
            time=_strip_tz(df_portable_dimm.index),
            fwhm=df_portable_dimm[portable_col].values,
        )
    )

    cam_time = _strip_tz(pd.to_datetime(df_cdb[cam_time_col]))
    src_cam = ColumnDataSource(
        data=dict(
            time=cam_time,
            fwhm=df_cdb[cam_fwhm_col].values,
            band=(
                df_cdb["band"].values
                if "band" in df_cdb.columns
                else [""] * len(df_cdb)
            ),
            airmass=(
                df_cdb["airmass"].values
                if "airmass" in df_cdb.columns
                else [np.nan] * len(df_cdb)
            ),
        )
    )

    # ── figure ──────────────────────────────────────────────────
    from bokeh.models import (
        BoxZoomTool,
        PanTool,
        ResetTool,
        SaveTool,
        WheelZoomTool,
    )

    wheel_x = WheelZoomTool(dimensions="width")
    xbox = BoxZoomTool(dimensions="width")
    ybox = BoxZoomTool(dimensions="height")

    p = figure(
        width=width,
        height=height,
        x_axis_type="datetime",
        y_range=Range1d(*y_range),
        y_axis_label="Zenith-corrected FWHM (arcsec)",
        title=title,
        tools=[PanTool(), xbox, ybox, wheel_x, ResetTool(), SaveTool()],
        active_drag=xbox,
        active_scroll=wheel_x,
    )

    # ── glyphs ──────────────────────────────────────────────────
    colors = {"tower": "#0072B2", "portable": "#D55E00", "cam": "#009E73"}

    r_tower = p.scatter(
        "time",
        "fwhm",
        source=src_tower,
        size=3,
        color=colors["tower"],
        alpha=0.6,
    )

    r_portable = p.scatter(
        "time",
        "fwhm",
        source=src_portable,
        size=3,
        color=colors["portable"],
        alpha=0.6,
    )

    r_cam = p.scatter(
        "time",
        "fwhm",
        source=src_cam,
        size=5,
        color=colors["cam"],
        alpha=0.7,
        marker="triangle",
    )

    # ── reference line at 1 arcsec ──────────────────────────────
    p.add_layout(
        Span(
            location=1.0,
            dimension="width",
            line_color="gray",
            line_dash="dashed",
            line_width=0.8,
        )
    )

    # ── hover tools (one per renderer for cleaner tooltips) ─────
    hover_dimm = HoverTool(
        renderers=[r_tower, r_portable],
        tooltips=[
            ("Time", "@time{%F %H:%M:%S}"),
            ("FWHM", "@fwhm{0.2f} arcsec"),
        ],
        formatters={"@time": "datetime"},
    )

    hover_cam = HoverTool(
        renderers=[r_cam],
        tooltips=[
            ("Time", "@time{%F %H:%M:%S}"),
            ("FWHM", "@fwhm{0.2f} arcsec"),
            ("Band", "@band"),
            ("Airmass", "@airmass{0.2f}"),
        ],
        formatters={"@time": "datetime"},
    )

    p.add_tools(hover_dimm, hover_cam)

    # ── legend ──────────────────────────────────────────────────
    legend = Legend(
        items=[
            LegendItem(label="Tower DIMM (sal=1)", renderers=[r_tower]),
            LegendItem(label="Portable DIMM (sal=2)", renderers=[r_portable]),
            LegendItem(label="LSSTCam median FWHM", renderers=[r_cam]),
        ],
        location="top_right",
    )

    legend.click_policy = "hide"
    legend.label_text_font_size = "9pt"
    legend.background_fill_alpha = 0.7
    p.add_layout(legend)

    # ── styling ─────────────────────────────────────────────────
    p.title.text_font_size = "13pt"
    p.xaxis.axis_label = "Time (UTC)"
    p.xaxis.axis_label_text_font_size = "10pt"
    p.yaxis.axis_label_text_font_size = "10pt"

    show(p)
    return p

In [ ]:
fwhm_timeline(df_tower_dimm, df_portable_dimm, df_lsstcam)

### Timeline using the wind threshold

In [ ]:
def _strip_tz(idx):
    """Strip timezone info to avoid Bokeh datetime64 warnings."""
    idx = pd.DatetimeIndex(idx)
    return idx.tz_localize(None) if idx.tz is not None else idx


def _make_dimm_source(df, fwhm_col, wind_col):
    """Build a ColumnDataSource from a DIMM dataframe."""
    return ColumnDataSource(
        data=dict(
            time=_strip_tz(df.index),
            fwhm=df[fwhm_col].values,
            wind_speed=df[wind_col].values,
        )
    )


def _make_cam_source(df, cam_time_col, cam_fwhm_col, wind_col):
    """Build a ColumnDataSource from a ConsDB dataframe."""
    return ColumnDataSource(
        data=dict(
            time=_strip_tz(pd.to_datetime(df[cam_time_col])),
            fwhm=df[cam_fwhm_col].values,
            wind_speed=df[wind_col].values,
            band=df["band"].values if "band" in df.columns else [""] * len(df),
            airmass=(
                df["airmass"].values if "airmass" in df.columns else [np.nan] * len(df)
            ),
        )
    )


def _build_panel(
    src_tower,
    src_portable,
    src_cam,
    x_range,
    y_range,
    tools,
    active_drag,
    active_scroll,
    width,
    height,
    title,
    show_xaxis,
):
    """Build a single FWHM panel figure."""

    p = figure(
        width=width,
        height=height,
        x_axis_type="datetime",
        x_range=x_range,
        y_range=Range1d(*y_range),
        y_axis_label="FWHM (arcsec)",
        tools=tools,
        active_drag=active_drag,
        active_scroll=active_scroll,
    )

    p.add_layout(Title(text=title, text_font_size="12pt"), "above")

    colors = {"tower": "#0072B2", "portable": "#D55E00", "cam": "#009E73"}

    r_tower = p.scatter(
        "time",
        "fwhm",
        source=src_tower,
        size=3,
        color=colors["tower"],
        alpha=0.6,
    )
    r_portable = p.scatter(
        "time",
        "fwhm",
        source=src_portable,
        size=3,
        color=colors["portable"],
        alpha=0.6,
    )
    r_cam = p.scatter(
        "time",
        "fwhm",
        source=src_cam,
        size=5,
        color=colors["cam"],
        alpha=0.7,
        marker="triangle",
    )

    # Reference line
    p.add_layout(
        Span(
            location=1.0,
            dimension="width",
            line_color="gray",
            line_dash="dashed",
            line_width=0.8,
        )
    )

    # Hovers
    p.add_tools(
        HoverTool(
            renderers=[r_tower, r_portable],
            tooltips=[
                ("Time", "@time{%F %H:%M:%S}"),
                ("FWHM", "@fwhm{0.2f} arcsec"),
                ("Wind", "@wind_speed{0.1f} m/s"),
            ],
            formatters={"@time": "datetime"},
        )
    )
    p.add_tools(
        HoverTool(
            renderers=[r_cam],
            tooltips=[
                ("Time", "@time{%F %H:%M:%S}"),
                ("FWHM", "@fwhm{0.2f} arcsec"),
                ("Band", "@band"),
                ("Airmass", "@airmass{0.2f}"),
                ("Wind", "@wind_speed{0.1f} m/s"),
            ],
            formatters={"@time": "datetime"},
        )
    )

    # Legend
    legend = Legend(
        items=[
            LegendItem(label="Tower DIMM", renderers=[r_tower]),
            LegendItem(label="Portable DIMM", renderers=[r_portable]),
            LegendItem(label="LSSTCam", renderers=[r_cam]),
        ],
        location="top_right",
    )
    legend.click_policy = "hide"
    legend.label_text_font_size = "9pt"
    legend.background_fill_alpha = 0.7
    p.add_layout(legend)

    # Hide x-axis on top panel
    if not show_xaxis:
        p.xaxis.visible = False

    return p


def fwhm_timeline_wind_split(
    df_tower_dimm: pd.DataFrame,
    df_portable_dimm: pd.DataFrame,
    df_cdb: pd.DataFrame,
    wind_threshold: float = 6.0,
    tower_col: str = "fwhm_z",
    portable_col: str = "fwhm_z",
    cam_time_col: str = "obs_end",
    cam_fwhm_col: str = "fwhm_zenith_500nm_median",
    wind_col_dimm: str = "wind_speed",
    wind_col_cam: str = "wind_speed",
    width: int = 1200,
    height: int = 350,
    y_range: tuple = (0, 4),
):
    """Two-panel FWHM timeline split by wind speed threshold.

    Parameters
    ----------
    df_tower_dimm, df_portable_dimm : pd.DataFrame
        DIMM data with DatetimeIndex, ``fwhm_z``, and ``wind_speed`` columns.
        Use merge_asof with ESS data to add wind_speed beforehand.
    df_cdb : pd.DataFrame
        LSSTCam ConsDB data with ``wind_speed`` from the exposure table.
    wind_threshold : float
        Wind speed cutoff in m/s.
    """

    # ── split by wind threshold ─────────────────────────────────
    def _split(df, col):
        mask = df[col] < wind_threshold
        return df[mask].copy(), df[~mask].copy()

    tower_lo, tower_hi = _split(
        df_tower_dimm.dropna(subset=[wind_col_dimm, tower_col]), wind_col_dimm
    )
    port_lo, port_hi = _split(
        df_portable_dimm.dropna(subset=[wind_col_dimm, portable_col]), wind_col_dimm
    )
    cam_lo, cam_hi = _split(
        df_cdb.dropna(subset=[wind_col_cam, cam_fwhm_col]), wind_col_cam
    )

    # ── build sources ───────────────────────────────────────────
    src_tower_lo = _make_dimm_source(tower_lo, tower_col, wind_col_dimm)
    src_tower_hi = _make_dimm_source(tower_hi, tower_col, wind_col_dimm)
    src_port_lo = _make_dimm_source(port_lo, portable_col, wind_col_dimm)
    src_port_hi = _make_dimm_source(port_hi, portable_col, wind_col_dimm)
    src_cam_lo = _make_cam_source(cam_lo, cam_time_col, cam_fwhm_col, wind_col_cam)
    src_cam_hi = _make_cam_source(cam_hi, cam_time_col, cam_fwhm_col, wind_col_cam)

    n_lo = len(tower_lo) + len(port_lo) + len(cam_lo)
    n_hi = len(tower_hi) + len(port_hi) + len(cam_hi)

    # ── shared tools (each panel needs its own instances) ───────
    def _make_tools():
        return [
            PanTool(),
            BoxZoomTool(dimensions="width"),
            BoxZoomTool(dimensions="height"),
            WheelZoomTool(dimensions="width"),
            ResetTool(),
            SaveTool(),
        ]

    # ── top panel: wind < threshold ─────────────────────────────
    tools_top = _make_tools()
    p_top = _build_panel(
        src_tower_lo,
        src_port_lo,
        src_cam_lo,
        x_range=DataRange1d(),
        y_range=y_range,
        tools=tools_top,
        active_drag=tools_top[1],  # xbox_zoom
        active_scroll=tools_top[3],  # wheel_x
        width=width,
        height=height,
        title=f"Wind < {wind_threshold} m/s  (N = {n_lo:,})",
        show_xaxis=False,
    )

    # ── bottom panel: wind >= threshold, shared x_range ─────────
    tools_bot = _make_tools()
    p_bot = _build_panel(
        src_tower_hi,
        src_port_hi,
        src_cam_hi,
        x_range=p_top.x_range,  # ← shared
        y_range=y_range,
        tools=tools_bot,
        active_drag=tools_bot[1],
        active_scroll=tools_bot[3],
        width=width,
        height=height,
        title=f"Wind ≥ {wind_threshold} m/s  (N = {n_hi:,})",
        show_xaxis=True,
    )

    p_bot.xaxis.axis_label = "Time (UTC)"

    layout = column(p_top, p_bot)
    show(layout)
    return p_top, p_bot

In [ ]:
# df_cdb already has wind_speed from ConsDB
p_top, p_bot = fwhm_timeline_wind_split(
    df_tower_wind,
    df_portable_wind,
    df_lsstcam,
    wind_threshold=wind_threshold,
)

### Correlation Plots - Raw

In [ ]:
# Ensure all are sorted by index
df_base = df_cdb[["obs_start", "fwhm_zenith_500nm_median"]].copy()
df_base["obs_start"] = pd.to_datetime(df_base["obs_start"])
df_base = df_base.set_index("obs_start").sort_index()
df_base.index = df_base.index.tz_localize("UTC")

df_t = df_tower_dimm[["fwhm_z"]].rename(columns={"fwhm_z": "tower_fwhm_z"}).sort_index()
df_p = (
    df_portable_dimm[["fwhm_z"]]
    .rename(columns={"fwhm_z": "portable_fwhm_z"})
    .sort_index()
)
df_w = df_ess_airflow[["mean_speed", "mean_direction"]].sort_index()

# Resample DIMM and wind to smooth out before matching
# NOTE: Resampling wind direction with .mean() is not strictly correct
#  for angular data (e.g., averaging 350° and 10° gives 180° instead of 0°).
#  Consider using circular statistics if precision matters.
df_t = df_t.resample("30s").mean().dropna()
df_p = df_p.resample("30s").mean().dropna()
df_w = df_w.resample("30s").mean().dropna()

# Merge all onto df_base with a tolerance window
tol = pd.Timedelta(seconds=30)

df_merged = pd.merge_asof(
    df_base, df_t, left_index=True, right_index=True, tolerance=tol, direction="nearest"
)
df_merged = pd.merge_asof(
    df_merged,
    df_p,
    left_index=True,
    right_index=True,
    tolerance=tol,
    direction="nearest",
)
df_merged = pd.merge_asof(
    df_merged,
    df_w,
    left_index=True,
    right_index=True,
    tolerance=tol,
    direction="nearest",
)

df_merged = df_merged.dropna()

In [ ]:
def triangle_plot(
    df, cols=None, figsize=(12, 12), s=3, alpha=0.25, color="steelblue", bins=30
):
    if cols is None:
        cols = df.columns.tolist()
    data = df[cols].dropna()
    n = len(cols)
    fig, axes = plt.subplots(n, n, figsize=figsize)

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
            elif i == j:
                ax.hist(data[cols[i]], bins=bins, alpha=alpha, color=color)
            else:
                ax.scatter(data[cols[j]], data[cols[i]], s=s, alpha=alpha, color=color)
                ax.grid(True, linestyle=":", alpha=0.3)
                r = data[cols[j]].corr(data[cols[i]])
                ax.annotate(
                    f"r={r:.2f}",
                    xy=(0.95, 0.95),
                    xycoords="axes fraction",
                    ha="right",
                    va="top",
                    fontsize=9,
                    bbox=dict(boxstyle="round", fc="white", alpha=0.8),
                )
                try:
                    x = data[cols[j]].astype(float).values
                    y = data[cols[i]].astype(float).values
                    xy = np.vstack([x, y])
                    kde = gaussian_kde(xy)
                    xi = np.linspace(x.min(), x.max(), 100)
                    yi = np.linspace(y.min(), y.max(), 100)
                    Xi, Yi = np.meshgrid(xi, yi)
                    Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
                    ax.contour(
                        Xi, Yi, Zi, levels=5, colors="k", linewidths=0.5, alpha=0.7
                    )
                except np.linalg.LinAlgError:
                    pass  # Skip if KDE fails (e.g., singular matrix)

            if i == n - 1:
                ax.set_xlabel(cols[j], fontsize=9)
            else:
                ax.set_xticklabels([])
            if j == 0:
                ax.set_ylabel(cols[i], fontsize=9)
            else:
                ax.set_yticklabels([])

    plt.tight_layout()
    return fig, axes

In [ ]:
fig, axes = triangle_plot(df_merged)
plt.show()

### Correlation Plots - Seeing Differences

In [ ]:
df_merged["tower_dimm_minus_portable_dimm_fwhm_z"] = (
    df_merged.tower_fwhm_z - df_merged.portable_fwhm_z
)
df_merged["lsstcam_minus_tower_dimm_fwhm_z"] = (
    df_merged.fwhm_zenith_500nm_median - df_merged.tower_fwhm_z
)
df_merged["lsstcam_minus_portable_dimm_fwhm_z"] = (
    df_merged.fwhm_zenith_500nm_median - df_merged.portable_fwhm_z
)

In [ ]:
plot_cols = [
    "tower_dimm_minus_portable_dimm_fwhm_z",
    "lsstcam_minus_tower_dimm_fwhm_z",
    "lsstcam_minus_portable_dimm_fwhm_z",
    "mean_speed",
]
df_plot = df_merged[plot_cols].dropna()

In [ ]:
fig, axes = triangle_plot(df_plot, color="firebrick")
plt.show()

### Timeline Plots with Differences

In [ ]:
def make_compass(size=120):

    p = figure(
        width=size,
        height=size,
        x_range=(-1.5, 1.5),
        y_range=(-1.5, 1.5),
        toolbar_location=None,
        min_border=0,
        match_aspect=True,
        sizing_mode="fixed",
    )

    p.axis.visible = False
    p.grid.visible = False
    p.outline_line_color = None

    # Arrows: N(up), E(right), S(down), W(left)
    head_kw = dict(size=8, line_color="green")

    for dx, dy, label, tx, ty in [
        (0, 1, "N", 0, 1.3),
        (1, 0, "E", 1.3, 0),
        (0, -1, "S", 0, -1.3),
        (-1, 0, "W", -1.3, 0),
    ]:
        p.add_layout(
            Arrow(
                end=OpenHead(**head_kw),
                line_color="green",
                x_start=0,
                y_start=0,
                x_end=dx,
                y_end=dy,
            )
        )
        p.add_layout(
            Label(
                x=tx,
                y=ty,
                text=label,
                text_align="center",
                text_baseline="middle",
                text_font_size="10pt",
                text_color="green",
            )
        )

    return p


def timeline_plot(
    df,
    diff_cols,
    wind_speed_col="mean_speed",
    wind_dir_col="mean_direction",
    width=1000,
    height=300,
    s=3,
    colors=None,
):
    if colors is None:
        colors = ["#0072B2", "#D55E00", "#009E73"]

    p1 = figure(
        width=width,
        height=height,
        x_axis_type="datetime",
        y_axis_label="ΔFWHM (arcsec)",
        tools="pan,xbox_zoom,xwheel_zoom,reset,save",
    )

    for col, color in zip(diff_cols, colors):
        p1.scatter(df.index, df[col], size=s, color=color, alpha=0.7, legend_label=col)

    p1.add_layout(
        Span(
            location=0,
            dimension="width",
            line_color="black",
            line_dash="dashed",
            line_width=0.5,
        )
    )
    p1.legend.click_policy = "hide"
    p1.legend.label_text_font_size = "8pt"

    p2 = figure(
        width=width,
        height=height,
        x_axis_type="datetime",
        x_range=p1.x_range,
        x_axis_label="Time",
        y_axis_label="Wind Speed (m/s)",
        tools="pan,xbox_zoom,xwheel_zoom,reset,save",
    )

    speed = df[wind_speed_col].values
    direction = np.deg2rad(270.0 - df[wind_dir_col].values)
    scale = 0.3
    dx = np.cos(direction) * speed * scale
    dy = np.sin(direction) * speed * scale

    x0 = df.index
    y0 = speed
    dx_ms = dx * 60_000
    x1 = x0 + pd.to_timedelta(dx_ms, unit="ms")
    y1 = y0 + dy

    p2.segment(x0=x0, y0=y0, x1=x1, y1=y1, color="green", line_width=1, alpha=0.7)
    p2.scatter(x0, y0, size=1, color="green", alpha=0.5)

    compass = make_compass()
    layout = column(p1, row(p2, compass, sizing_mode="stretch_width"))
    show(layout)
    return p1, p2

In [ ]:
p1, p2 = timeline_plot(
    df_merged,
    diff_cols=[
        "tower_dimm_minus_portable_dimm_fwhm_z",
        "lsstcam_minus_tower_dimm_fwhm_z",
        "lsstcam_minus_portable_dimm_fwhm_z",
    ],
)

## Polar Plots

In [ ]:
def polar_seeing_plot(
    df,
    seeing_col,
    direction_col="mean_direction",
    speed_col="mean_speed",
    figsize=(8, 8),
    s=20,
    alpha=0.5,
    vmin=None,
    vmax=None,
):
    data = df[[seeing_col, direction_col, speed_col]].dropna()

    theta = np.deg2rad(data[direction_col].values)
    r = data[seeing_col].values
    speed = data[speed_col].values

    # Discrete colormap in 1 m/s steps
    if vmin is None:
        vmin = int(np.floor(speed.min()))
    if vmax is None:
        vmax = int(np.ceil(speed.max()))

    bounds = np.arange(vmin, vmax + 1, 2)
    norm = mcolors.BoundaryNorm(bounds, ncolors=256)

    full_viridis = plt.cm.viridis
    truncated_viridis = mcolors.LinearSegmentedColormap.from_list(
        "viridis_trunc", full_viridis(np.linspace(0.2, 1.0, 256))
    )

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "polar"})
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    sc = ax.scatter(
        theta, r, c=speed, cmap=truncated_viridis, norm=norm, s=s, alpha=alpha
    )

    cbar = plt.colorbar(sc, ax=ax, pad=0.1, ticks=bounds)
    cbar.set_label("Wind Speed (m/s)")

    ax.set_ylabel(seeing_col, labelpad=30)
    ax.set_title(f"Seeing vs Wind Direction\n({seeing_col})", pad=20)

    plt.tight_layout()
    return fig, ax

### Tower Dimm Seeing

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col="tower_fwhm_z", alpha=1)

### Portable DIMM Seeing

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col="portable_fwhm_z", alpha=1)

### LSSTCam Median FWHM at Zenith and 500 nm

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col="fwhm_zenith_500nm_median", alpha=1)

### Tower DIMM - Portable DIMM

In [ ]:
fig, ax = polar_seeing_plot(
    df_merged, seeing_col="tower_dimm_minus_portable_dimm_fwhm_z", alpha=1
)

### LSSTCam - Tower DIMM

In [ ]:
fig, ax = polar_seeing_plot(
    df_merged, seeing_col="lsstcam_minus_tower_dimm_fwhm_z", alpha=1
)

### LSSTCam - Portable DIMM

In [ ]:
fig, ax = polar_seeing_plot(
    df_merged, seeing_col="lsstcam_minus_portable_dimm_fwhm_z", alpha=1
)

## Binned heatmap (2D histogram in polar coordinates)

In [ ]:
def polar_binned_plot(
    df,
    seeing_col,
    direction_col="mean_direction",
    speed_col="mean_speed",
    n_angle_bins=36,
    n_radial_bins=20,
    figsize=(8, 8),
    cmap="viridis",
):
    data = df[[seeing_col, direction_col, speed_col]].dropna()

    angle_bins = np.linspace(0, 360, n_angle_bins + 1)
    radial_bins = np.linspace(
        data[seeing_col].min(), data[seeing_col].max(), n_radial_bins + 1
    )

    data["angle_bin"] = pd.cut(data[direction_col], bins=angle_bins, labels=False)
    data["radial_bin"] = pd.cut(data[seeing_col], bins=radial_bins, labels=False)

    grouped = data.groupby(["angle_bin", "radial_bin"])[speed_col].mean().reset_index()

    theta = np.deg2rad((angle_bins[:-1] + angle_bins[1:]) / 2)
    r = (radial_bins[:-1] + radial_bins[1:]) / 2
    dtheta = np.deg2rad(360 / n_angle_bins)
    dr = radial_bins[1] - radial_bins[0]

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "polar"})
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    for _, row in grouped.iterrows():
        ai, ri = int(row["angle_bin"]), int(row["radial_bin"])
        ax.bar(
            theta[ai],
            dr,
            width=dtheta,
            bottom=r[ri] - dr / 2,
            color=plt.cm.viridis(
                plt.Normalize(data[speed_col].min(), data[speed_col].max())(
                    row[speed_col]
                )
            ),
            alpha=0.8,
            edgecolor="none",
        )

    sm = plt.cm.ScalarMappable(
        cmap=cmap, norm=plt.Normalize(data[speed_col].min(), data[speed_col].max())
    )
    cbar = plt.colorbar(sm, ax=ax, pad=0.1)
    cbar.set_label(f"Mean Wind Speed (m/s)")
    ax.set_title(f"{seeing_col}", pad=20)
    plt.tight_layout()
    return fig, ax

### Tower Dimm Seeing

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col="tower_fwhm_z")

### Portable DIMM Seeing

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col="portable_fwhm_z")

### LSSTCam Median FWHM at Zenith and 500 nm

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col="lsstcam_minus_tower_dimm_fwhm_z")

### Tower DIMM - Portable DIMM

In [ ]:
fig, ax = polar_binned_plot(
    df_merged, seeing_col="tower_dimm_minus_portable_dimm_fwhm_z"
)

### LSSTCam - Tower DIMM

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col="lsstcam_minus_tower_dimm_fwhm_z")

### LSSTCam - Portable DIMM

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col="lsstcam_minus_portable_dimm_fwhm_z")

## Contour plot in polar coordinates


In [ ]:
def polar_contour_plot(
    df,
    seeing_col,
    direction_col="mean_direction",
    speed_col="mean_speed",
    figsize=(8, 8),
    n_angle=72,
    n_radial=50,
):
    data = df[[seeing_col, direction_col, speed_col]].dropna()

    theta_grid = np.linspace(0, 2 * np.pi, n_angle)
    r_grid = np.linspace(data[seeing_col].min(), data[seeing_col].max(), n_radial)
    T, R = np.meshgrid(theta_grid, r_grid)

    from scipy.stats import binned_statistic_2d

    stat, _, _, _ = binned_statistic_2d(
        np.deg2rad(data[direction_col].values),
        data[seeing_col].values,
        data[speed_col].values,
        statistic="mean",
        bins=[theta_grid, r_grid],
    )

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "polar"})
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    cs = ax.pcolormesh(T, R, stat.T, cmap="viridis", shading="auto")
    cbar = plt.colorbar(cs, ax=ax, pad=0.1)
    cbar.set_label("Mean Wind Speed (m/s)")
    ax.set_title(f"{seeing_col}", pad=20)
    plt.tight_layout()
    return fig, ax

In [ ]:
fig, ax = polar_contour_plot(df_merged, seeing_col="tower_fwhm_z")